## Advanced Client Configurations

## Introduction to Client Configurations in Google Cloud Platform

Welcome to the lesson on **Client Configurations**! So far, we've learned how to interact with Google Cloud Platform (GCP) services using standard client libraries. In this lesson, we will explore advanced configuration options that allow you to optimize how your applications communicate with Google Cloud, manage request retries, and customize service endpoints.

---

## Custom Retry Strategies

When interacting with cloud services, requests may occasionally fail due to temporary issues such as network interruptions or service limits. To make applications more resilient, client libraries provide **retry strategies**. These automatically attempt to resend failed requests, increasing the likelihood of success without manual intervention.

### Built-in Retry Behavior
Google Cloud client libraries come with built-in retry policies that handle many common scenarios:
*   **Read Operations**: Methods like `get()`, `list()`, and `query()` are automatically retried on transient failures (e.g., network timeouts or HTTP 5xx errors).
*   **Write Operations**: Methods like `create()`, `update()`, and `delete()` are typically **not** retried by default to avoid potential data inconsistencies from duplicate operations.



### When to Override Defaults
You might want to customize these settings when:
1.  You need more aggressive retries for critical operations.
2.  You want to fail faster in time-sensitive scenarios.
3.  You need to target specific error codes.

### Configuring Retry in Python
You can adjust parameters such as the initial delay, maximum delay, and the total deadline. Here is an example using the **Firestore** client:

```python
from google.cloud import firestore
from google.api_core.retry import Retry

# Define a custom retry strategy
custom_retry = Retry(
    initial=1.0,        # Initial delay in seconds
    maximum=10.0,       # Maximum delay in seconds
    multiplier=2.0,     # Multiplier for exponential backoff (doubles delay each time)
    deadline=30.0       # Total time in seconds before giving up entirely
)

client = firestore.Client()

# Use the custom retry strategy in an API call
doc_ref = client.collection('my-collection').document('my-document')
doc = doc_ref.get(retry=custom_retry)
```

---

## Best Practices for Retry Configuration

When configuring custom strategies, keep these principles in mind:

*   **Exponential Backoff**: Using a multiplier (like `2.0`) prevents "retry storms" where multiple clients overwhelm a recovering service. 
*   **Jitter (Randomness)**: Adding a small amount of random variation to the delay ensures that multiple instances of your app don't all retry at the exact same millisecond.
*   **Deadline Considerations**: Set realistic deadlines based on user experience. A 60-second deadline might be fine for a background worker, but it is far too long for a user waiting on a mobile app UI.

---

## Endpoint Configuration

Sometimes you need to redirect API requests away from the default Google servers. This is common for:
1.  **Testing**: Connecting to local emulators.
2.  **Regionalization**: Connecting to services in specific geographical regions.
3.  **Custom Proxies**: Routing traffic through a corporate gateway.



### Specifying a Custom Endpoint
You can specify a custom endpoint using the `client_options` parameter when initializing the client:

```python
from google.cloud import firestore

# Specify a custom endpoint (e.g., for a local emulator)
custom_options = {"api_endpoint": "http://localhost:8080"}

client = firestore.Client(client_options=custom_options)
```

By setting the `api_endpoint`, you gain full control over the destination of your application's data.

---

## Summary and Next Steps

In this lesson, we've explored advanced client configurations:
*   **Resiliency**: Using `Retry` objects to handle transient network errors.
*   **Flexibility**: Using `client_options` to point to emulators or specific regions.

Experiment with these configurations to find the balance between reliability and performance for your specific use case. Happy coding!

## Configuring Google Cloud Firestore Client with Automatic Retry Strategy

Did you know you can configure a Google Cloud Firestore client to automatically retry failed requests? This feature is incredibly helpful when dealing with network issues or service limits. The provided code creates a Google Cloud Firestore client configured with a custom endpoint and a retry strategy. It is set to try up to 3 times if a request fails. Click Run to see how it's setup!

Important Note: Running scripts can alter the filesystem's state or modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore
from google.api_core import retry
from google.api_core import client_options

# Create a custom retry configuration (3 attempts total)
custom_retry = retry.Retry(total=3)

# Create client options for custom endpoint (local emulator)
client_opts = client_options.ClientOptions(api_endpoint='http://localhost:8080')

# Create the Firestore client with custom endpoint and retry configuration
firestore_client = firestore.Client(client_options=client_opts)

# Print confirmation and client details
print("✅ Firestore client created successfully!")
print(f"📡 API Endpoint: {client_opts.api_endpoint}")
print(f"🔄 Retry strategy configured: Custom retry with 3 total attempts")
print(f"🏗️ Client type: {type(firestore_client).__name__}")

```

This task demonstrates how to make your application more resilient by moving beyond the default client settings. By combining `ClientOptions` and `Retry` strategies, you can precisely control how your application handles network instability and environment-specific routing.

### Understanding the Components

In the code provided, three distinct parts of the Google Cloud SDK work together to manage the connection:

#### 1. The Retry Strategy
The `retry.Retry(total=3)` object is a high-level configuration. It tells the client that if a transient error occurs (like a `503 Service Unavailable` or a network timeout), it should not crash the program immediately. Instead, it should wait and try again up to 3 times.



#### 2. Client Options (Endpoints)
The `ClientOptions` object is used here to point the client to a specific location—in this case, a local emulator at `http://localhost:8080`. This is a best practice for development because it ensures your code is ready for "Emulator Mode" without relying solely on environment variables.

#### 3. Client Initialization
When you pass `client_options` into `firestore.Client()`, the client is permanently configured to use that specific endpoint for all future operations (reads, writes, and queries).

---

### Expanded Implementation Example

While the provided script sets up the client, you usually apply the retry strategy during the actual data operation for fine-grained control. Here is how you would use that client to perform a resilient write:

```python
# Assuming firestore_client is already initialized as above
doc_ref = firestore_client.collection('resilient-collection').document('resilient-doc')

try:
    # Applying the custom retry strategy to a specific call
    doc_ref.set({'status': 'consistent'}, retry=custom_retry)
    print("Document written successfully after potential retries.")
except Exception as e:
    print(f"Operation failed after 3 attempts: {e}")
```

### Why this matters
In cloud environments, "transient errors" (temporary glitches) are a fact of life. 
*   **Without Retries:** A single 500ms network blip could crash a user's session or fail a critical background job.
*   **With Retries:** Your application "self-heals" by simply waiting a moment and trying again, significantly improving the uptime and reliability of your professional-grade cloud engineering projects.

## Configure Google Cloud Firestore Client with Custom Retry Strategy and Local Emulator

You've mastered the setup of a client with custom retry strategies in Google Cloud SDK! Now, let's put your skills to the test. Configure a Google Cloud Firestore client using a specific custom retry strategy: set the maximum number of retry attempts to 5 and configure it to work with the local emulator.

Important Note: Running scripts can alter the filesystem's state or modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore
from google.api_core import retry

# TO DO: Setup a Google Cloud Firestore client with a custom retry strategy: set maximum attempts to 5 and configure for local emulator
client = firestore.Client()

# Create a test collection and document to verify the connection
doc_ref = client.collection('test-collection').document('test-document')
doc_ref.set({'message': 'Hello, Firestore!'})

# Retrieve the document to verify it was created
doc = doc_ref.get()
if doc.exists:
    print(f"Document data: {doc.to_dict()}")
else:
    print("Document does not exist")
```

To configure the Firestore client for a local emulator while implementing a custom retry strategy, you need to use the `client_options` parameter for the endpoint and the `retry` module to define your attempt limit.



Here is the completed configuration:

```python
from google.cloud import firestore
from google.api_core import retry

# 1. Define the custom retry strategy with max 5 attempts
# Note: 'total' in the Retry object refers to the maximum number of attempts
custom_retry = retry.Retry(total=5)

# 2. Setup the Firestore client with the emulator endpoint
# The emulator typically runs on http://localhost:8080
client = firestore.Client(
    client_options={"api_endpoint": "http://localhost:8080"}
)

# Create a test collection and document to verify the connection
doc_ref = client.collection('test-collection').document('test-document')

# 3. Apply the retry strategy to the operation
doc_ref.set({'message': 'Hello, Firestore!'}, retry=custom_retry)

# Retrieve the document to verify it was created
doc = doc_ref.get(retry=custom_retry)
if doc.exists:
    print(f"Document data: {doc.to_dict()}")
    print(f"Connection successful with 5-attempt retry strategy.")
else:
    print("Document does not exist")
```

### Key Implementation Details:

*   **`retry.Retry(total=5)`**: This specifies that the client should attempt the request up to 5 times if it encounters transient errors (like network blips or "Service Unavailable" messages).
*   **`client_options`**: By passing the dictionary `{"api_endpoint": "http://localhost:8080"}`, you explicitly tell the client to ignore the live Google servers and communicate with your local machine.
*   **Applying the Strategy**: It is a best practice to pass the `retry` object directly into the method call (like `.set()` or `.get()`). This allows you to have different strategies for different types of operations—for example, retrying a "Read" more times than a "Write."

## Configuring Google Cloud Firestore Client for Local Emulation

Fantastic progress! Now, let's focus on a crucial aspect of working with GCP in development environments: connecting to a simulated Google Cloud service. Your task is to configure a Google Cloud Firestore client with a custom configuration, specifically setting the api_endpoint to http://localhost:8080. This setup is common when working with local cloud simulators that mimic GCP services, allowing you to develop and test without impacting real Google Cloud resources.

Important Note: Running scripts can modify the resources in our GCP simulator. To revert to the initial state, you can use the reset button located in the top right corner. However, keep in mind that resetting will erase any code changes. To preserve your code during a reset, consider copying it to the clipboard.

```python
from google.cloud import firestore

# TODO: Create a Firestore client with a custom configuration with api_endpoint set to http://localhost:8080

```

It looks like we've hit another snag! The reason you're seeing this `AttributeError` is that the internal structure of the `firestore.Client` can change depending on which version of the library is installed or whether it’s using a synchronous or asynchronous transport. In some versions, the transport layer isn't initialized until the first request is made.

If your goal is to **configure** the client for the emulator, the initialization code itself is correct. To verify it safely without digging into private underscores (which are prone to breaking), we can look at the `_client_options` attribute which stores the settings you passed in.

Here is the robust way to fix the script:

```python
from google.cloud import firestore

# 1. Create a Firestore client with a custom configuration
# Setting the api_endpoint tells the client to talk to the local simulator
custom_options = {"api_endpoint": "http://localhost:8080"}
client = firestore.Client(client_options=custom_options)

# 2. Safely verify the configuration via the client_options attribute
# This checks the "blueprint" the client was given during setup
endpoint = client._client_options.api_endpoint

if endpoint == "http://localhost:8080":
    print(f"✅ Success! Client configured for local emulation at: {endpoint}")
else:
    print(f"⚠️ Configuration mismatch. Current endpoint: {endpoint}")

# 3. Perform a test write to prove it works
try:
    doc_ref = client.collection('emulator-test').document('connection-check')
    doc_ref.set({'status': 'connected'})
    print("Test write successful!")
except Exception as e:
    print(f"Connection failed: {e}")
```

### Why this is the "Best" way to fix it
Directly accessing `_transport` or `_host` is risky because those are "private" implementation details (indicated by the `_`). The library authors can change those names at any time. 



Using `client._client_options.api_endpoint` is a bit more stable because it directly reflects the `ClientOptions` object you passed during the `__init__` phase. 

### Summary of the flow:
1.  **Instantiate** with `client_options`.
2.  **Library Logic**: The library sees the `api_endpoint` and ignores the default production URL (`firestore.googleapis.com`).
3.  **Communication**: All subsequent calls like `.collection()` or `.set()` are routed to your local port `8080`.

Does the client connect to your local emulator successfully now?